# Solution 44: Supervised instruction fine-tuning

Continue the **same model weights** saved by assignment 43. Do not initialize a new model for this stage. Train on prompt/desired-answer conversations with shifted next-token CE, but mask system/user text, role prefixes and padding so only assistant content and end markers contribute to the loss.

The offline SFT fixture has 13 short training dialogues. For a larger run, manually download the [10.5 MB No Robots instruction file](https://huggingface.co/datasets/HuggingFaceH4/no_robots/resolve/main/data/train-00000-of-00001.parquet?download=true), convert it as described in [the data guide](../datasets/llm/README.md), then rerun this notebook. A tiny base model will not gain broad language ability from SFT alone.

Implement `SFTTrainer.encode_messages(messages, tokenizer)` and `SFTTrainer.reply(model, tokenizer, messages, max_new_tokens)`. You may reuse your assignment 43 `LLMTrainer.train_step` and `generate`, plus basic PyTorch utilities. The prepared `TextDataset` calls your encoder for every conversation.

- Encoding: begin with BOS; append `role + ': '`, content and SEP per turn, except the final assistant turn ends with EOS. Labels have `-100` for BOS, all role prefixes, system/user content and separators, and padding. Supervise assistant content and its SEP/EOS. Reject an empty conversation, unsupported roles, or one not ending with assistant.
- Reply: `messages` ends with a user turn. Encode with the same `assistant: ` prompt used in training, greedily generate up to the budget, return decoded *new* assistant text (exclude EOS and the prompt). Preserve the model's original train/eval mode.

The notebook loads `artifacts/llm/base_reference.pt`, trains all model parameters, saves `artifacts/llm/instruction_reference.pt`, and demonstrates a reply. The judge checks the masks and formatting with fixed inputs; it does not require one exact open-ended sentence. Run assignment 45 after this notebook.


In [ ]:
from pathlib import Path
from dataclasses import asdict
import torch
from torch.utils.data import DataLoader
from torch_judge.capstone import LLMConfig, ByteTokenizer, TextDataset, read_jsonl, deterministic
from mini_llm_reference import MiniLLM
from pretrain_trainer_reference import LLMTrainer


In [ ]:
%%writefile sft_trainer_reference.py
import torch
from pretrain_trainer_reference import LLMTrainer

class SFTTrainer:
    @staticmethod
    def encode_messages(messages, tokenizer):
        if not messages or messages[-1]['role'] != 'assistant':
            raise ValueError('Conversation must end with assistant')
        ids, labels = [tokenizer.bos_id], [-100]
        for index, message in enumerate(messages):
            role, content = message['role'], message['content']
            if role not in ('system', 'user', 'assistant') or not isinstance(content, str):
                raise ValueError('Invalid role or content')
            prefix = tokenizer.encode(role + ': ')
            body = tokenizer.encode(content)
            end = tokenizer.eos_id if index == len(messages)-1 else tokenizer.sep_id
            ids.extend(prefix + body + [end])
            labels.extend([-100] * len(prefix))
            labels.extend(body + [end] if role == 'assistant' else [-100] * (len(body)+1))
        return ids, labels

    @staticmethod
    def reply(model, tokenizer, messages, max_new_tokens=48):
        if not messages or messages[-1]['role'] != 'user':
            raise ValueError('Prompt must end with user')
        ids = tokenizer.prompt(messages)
        parameter = next(model.parameters(), None)
        device = parameter.device if parameter is not None else 'cpu'
        prompt = torch.tensor([ids], dtype=torch.long, device=device)
        output = LLMTrainer.generate(model, prompt, max_new_tokens)
        return tokenizer.decode(output[0, len(ids):].tolist())


In [ ]:
from sft_trainer_reference import SFTTrainer
from torch_judge import check
check('llm_sft')


In [ ]:
DATA = Path('datasets/llm/sft')
is_demo = not (DATA / 'train.jsonl').exists()
train_path = DATA / ('demo_train.jsonl' if is_demo else 'train.jsonl')
valid_path = DATA / ('demo_validation.jsonl' if is_demo else 'validation.jsonl')
base_path = Path('artifacts/llm/base_reference.pt')
if not base_path.exists():
    raise FileNotFoundError('Run assignment 43 first to create artifacts/llm/base_reference.pt')
base = torch.load(base_path, map_location='cpu', weights_only=True)
assert base['tokenizer'] == 'utf8_byte_v1'
config = LLMConfig(**base['config'])
train_rows, valid_rows = read_jsonl(train_path), read_jsonl(valid_path)
encoder = lambda record, tok: SFTTrainer.encode_messages(record['messages'], tok)
train_data = TextDataset(train_rows, config.max_seq_len, encoder=encoder)
valid_data = TextDataset(valid_rows, config.max_seq_len, encoder=encoder)
print(f'SFT: {len(train_data)}/{len(train_rows)} train records retained, {len(valid_data)} validation records')
train_loader = DataLoader(train_data, batch_size=4, shuffle=False, num_workers=0)
valid_loader = DataLoader(valid_data, batch_size=4, shuffle=False, num_workers=0)


def evaluate_sft(model, loader):
    was_training = model.training
    model.eval()
    total = count = 0
    try:
        with torch.no_grad():
            for ids, labels in loader:
                n = int((labels[:, 1:] != -100).sum())
                total += LLMTrainer.loss(model(ids), labels).item() * n
                count += n
    finally:
        model.train(was_training)
    return total / count


def run_sft(epochs=12, max_steps=None):
    with deterministic(2027):
        model = MiniLLM(config).cpu()
        model.load_state_dict(base['model_state'])
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=.01)
        history, steps = [], 0
        for epoch in range(epochs):
            for ids, labels in train_loader:
                LLMTrainer.train_step(model, optimizer, [(ids, labels)], max_norm=1.0)
                steps += 1
                if max_steps is not None and steps >= max_steps:
                    break
            row = (evaluate_sft(model, train_loader), evaluate_sft(model, valid_loader))
            history.append(row)
            print(f'SFT epoch {epoch+1}: train={row[0]:.4f}, validation={row[1]:.4f}')
            if max_steps is not None and steps >= max_steps:
                break
        return model, optimizer, history

model, optimizer, history = run_sft(epochs=12 if is_demo else 1)


In [ ]:
checkpoint_path = Path('artifacts/llm/instruction_reference.pt')
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save({'config': asdict(config), 'model_state': model.state_dict(),
            'tokenizer': 'utf8_byte_v1', 'stage': 'sft'}, checkpoint_path)
print('Saved:', checkpoint_path)
print('Reply:', SFTTrainer.reply(model, ByteTokenizer(), [{'role':'user','content':'Hi'}]))
